# Loyalty Program Impact

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
df_order_items = spark.read.table('global_partner.silver.order_items')
df_order_item_options = spark.read.table('global_partner.silver.order_item_options')
df_date_dim = spark.read.table('global_partner.bronze.date_dim')

In [0]:
df_order_items.printSchema()

In [0]:
df_loyalty = df_order_items.select('CREATION_TIME_UTC','USER_ID','ITEM_PRICE','ORDER_ID','IS_LOYALTY').distinct()
df_loyalty = (df_loyalty.groupBy('IS_LOYALTY','USER_ID').agg(
                                         F.round(F.avg('ITEM_PRICE'),2).alias('Average_spend'),
                                         F.round(F.sum('ITEM_PRICE'),2).alias('Lifetime_value')))
df_loyalty.display()

# Repeat Orders

In [0]:
df_repeat_items = df_order_items.select('USER_ID','ITEM_PRICE','ORDER_ID','IS_LOYALTY','RESTAURANT_ID','ITEM_NAME').distinct()
df_repeat_items = (df_repeat_items.groupBy('USER_ID','IS_LOYALTY','RESTAURANT_ID','ITEM_NAME')
                   .agg(F.count('ITEM_NAME').alias('Order_cnt'))
                   .orderBy('USER_ID','IS_LOYALTY','RESTAURANT_ID','ITEM_NAME'))
df_repeat_items.display()

In [0]:
df_store = df_order_items.select('ITEM_PRICE','ORDER_ID','RESTAURANT_ID').distinct()
df_store = df_store.groupBy('RESTAURANT_ID').agg(F.round(F.sum('ITEM_PRICE'),2).alias('total_revenue')).orderBy(F.col('total_revenue').desc())
df_store.display()